In [1]:
!nvidia-smi

Mon Mar  2 00:14:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import tensorflow as tf
import time

gpus = tf.config.list_physical_devices('GPU')
print(f"Detected GPUs: {len(gpus)}")

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data()
x_train = tf.cast(x_train, tf.float32) / 255.0
y_train = tf.cast(y_train, tf.int32)

BATCH_SIZE = 256

train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_dataset = train_dataset.shuffle(10000).batch(BATCH_SIZE, drop_remainder=True)

with tf.device('/GPU:0'):
    model_part1 = tf.keras.Sequential([
        tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2))
    ])

with tf.device('/GPU:1'):
    model_part2 = tf.keras.Sequential([
        tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D((2, 2)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(512, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(100)
    ])

optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

train_loss_tracker = tf.keras.metrics.Mean(name='train_loss')
train_acc_tracker = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')

@tf.function
def model_parallel_train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        with tf.device('/GPU:0'):
            mid_output = model_part1(x_batch, training=True)
            
        with tf.device('/GPU:1'):
            final_output = model_part2(mid_output, training=True)
            loss = loss_fn(y_batch, final_output)
            
    trainable_vars = model_part1.trainable_variables + model_part2.trainable_variables
    grads = tape.gradient(loss, trainable_vars)
    
    optimizer.apply_gradients(zip(grads, trainable_vars))
    
    train_loss_tracker.update_state(loss)
    train_acc_tracker.update_state(y_batch, final_output)

EPOCHS = 100

for epoch in range(EPOCHS):
    start_time = time.time()
    
    train_loss_tracker.reset_state()
    train_acc_tracker.reset_state()
    
    for step, (x_batch, y_batch) in enumerate(train_dataset):
        model_parallel_train_step(x_batch, y_batch)
        
        if step % 50 == 0:
            print(f"  Step {step}: Loss = {train_loss_tracker.result():.4f}, "
                  f"Accuracy = {train_acc_tracker.result():.4f}")

    end_time = time.time()
    print(f"--- Epoch {epoch + 1} Completed (Time: {end_time - start_time:.2f} sec) ---")
    print(f"Epoch {epoch + 1} Result -> Average Loss: {train_loss_tracker.result():.4f}, "
          f"Average Accuracy: {train_acc_tracker.result():.4f}\n")

2026-03-02 11:12:48.018541: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772449968.264691      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772449968.339659      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772449968.910650      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772449968.910703      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772449968.910705      55 computation_placer.cc:177] computation placer alr

Detected GPUs: 2
169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


I0000 00:00:1772450002.951008      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1772450002.956933      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1772450007.043059     124 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1772450008.195217     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


  Step 0: Loss = 4.6168, Accuracy = 0.0039
  Step 50: Loss = 4.5022, Accuracy = 0.0214
  Step 100: Loss = 4.3663, Accuracy = 0.0350
  Step 150: Loss = 4.2378, Accuracy = 0.0504
--- Epoch 1 Completed (Time: 10.78 sec) ---
Epoch 1 Result -> Average Loss: 4.1414, Average Accuracy: 0.0642

  Step 0: Loss = 3.6629, Accuracy = 0.1055
  Step 50: Loss = 3.6643, Accuracy = 0.1385
  Step 100: Loss = 3.5969, Accuracy = 0.1511
  Step 150: Loss = 3.5304, Accuracy = 0.1631
--- Epoch 2 Completed (Time: 5.12 sec) ---
Epoch 2 Result -> Average Loss: 3.4845, Average Accuracy: 0.1725

  Step 0: Loss = 3.1332, Accuracy = 0.2266
  Step 50: Loss = 3.2235, Accuracy = 0.2198
  Step 100: Loss = 3.1776, Accuracy = 0.2278
  Step 150: Loss = 3.1359, Accuracy = 0.2342
--- Epoch 3 Completed (Time: 5.13 sec) ---
Epoch 3 Result -> Average Loss: 3.1026, Average Accuracy: 0.2397

  Step 0: Loss = 2.8166, Accuracy = 0.3164
  Step 50: Loss = 2.9209, Accuracy = 0.2741
  Step 100: Loss = 2.8891, Accuracy = 0.2790
  Step 15